In [ ]:
from pymongo import MongoClient
from openai import AzureOpenAI
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["yelp_business"]
collection = db["business"]

business_docs = list(collection.find())  # 你也可以用分页处理
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o-mini"
def is_located_in_indianapolis(description):
    prompt = (
        f"Does the following business description clearly indicate that the business is located in Indianapolis?\n\n"
        f"Description: {description}\n\n"
        "Please answer only 'yes' or 'no'."
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that determines the city location of a business from its description."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower() == "yes"
    except Exception as e:
        print(f"❌ GPT error: {e}")
        return False
indianapolis_businesses = []

for i, biz in enumerate(business_docs):
    desc = biz.get("description", "")
    if not desc:
        continue

    result = is_located_in_indianapolis(desc)
    if result:
        indianapolis_businesses.append(biz)
        print(f"✅ Found: Business #{i}, {desc} → Indianapolis")
    else:
        print(f"❌ Skip: Business #{i} → Not in Indianapolis")
df_indianapolis = pd.DataFrame(indianapolis_businesses)
print(f"\n Retained {len(df_indianapolis)} businesses in Indianapolis")

❌ Skip: Business #0 → Not in Indianapolis
❌ Skip: Business #1 → Not in Indianapolis
❌ Skip: Business #2 → Not in Indianapolis
❌ Skip: Business #3 → Not in Indianapolis
❌ Skip: Business #4 → Not in Indianapolis
❌ Skip: Business #5 → Not in Indianapolis
❌ Skip: Business #6 → Not in Indianapolis
❌ Skip: Business #7 → Not in Indianapolis
✅ Found: Business #8, Located at 5000 W 96th St in Indianapolis, IN, this establishment offers a diverse selection of Antiques, Shopping, Home Services, and Lighting Fixtures & Equipment for all your home and decorative needs. → Indianapolis
❌ Skip: Business #9 → Not in Indianapolis
❌ Skip: Business #10 → Not in Indianapolis
❌ Skip: Business #11 → Not in Indianapolis
❌ Skip: Business #12 → Not in Indianapolis
❌ Skip: Business #13 → Not in Indianapolis
❌ Skip: Business #14 → Not in Indianapolis
❌ Skip: Business #15 → Not in Indianapolis
❌ Skip: Business #16 → Not in Indianapolis
❌ Skip: Business #17 → Not in Indianapolis
❌ Skip: Business #18 → Not in Indian

In [10]:
import duckdb

# 连接 DuckDB 数据库
con = duckdb.connect("../query_dataset/yelp_user.db")

# 列出所有表
tables = con.execute("SHOW TABLES").fetchall()

print("📋 数据库中包含的表：")
for t in tables:
    print("-", t[0])

df_review = con.execute("SELECT * FROM review").fetchdf()
df_tip = con.execute("SELECT * FROM tip").fetchdf()
df_user = con.execute("SELECT * FROM user").fetchdf()
con.close()

📋 数据库中包含的表：
- review
- tip
- user


In [13]:
import json
import duckdb
from pymongo import MongoClient
from openai import AzureOpenAI

deployment_name = "gpt-4o-mini"
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com"
)

# ========== Step 1: 加载两个 DuckDB 表中的 ID ==========
# business_ref from yelp_user.db
con_duck = duckdb.connect("../query_dataset/yelp_user.db")
business_refs = con_duck.execute("SELECT DISTINCT business_ref FROM review").fetchdf()["business_ref"].dropna().tolist()

# ========== Step 2: 从 MongoDB 读取 business_id ==========
client_mongo = MongoClient("mongodb://localhost:27017/")
business_ids = client_mongo["yelp_business"]["business"].distinct("business_id")

# ========== Step 2: 用 GPT 识别它们之间的映射规则 ==========
def get_mapping_rule(business_ids, business_refs):
    prompt = (
        "You are given two complete ID columns from two different datasets:\n"
        f"- The first column is `business_ref` from a review dataset: {json.dumps(business_refs, indent=2)}\n"
        f"- The second column is `business_id` from a business metadata dataset: {json.dumps(business_ids, indent=2)}\n\n"
        "Each business_ref corresponds to a business_id, but the mapping rule is not provided.\n"
        "Determine the mapping relationship between the two sets of IDs.\n"
        "Please respond with:\n"
        "1. The exact mapping rule in plain English\n"
        "2. An explanation of why you think this rule holds."
    )
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a data engineer analyzing two ID columns from different datasets to infer a mapping rule."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

mapping_rule_explanation = get_mapping_rule(business_ids, business_refs)
print("\n🧠 GPT-inferred mapping rule:\n")
print(mapping_rule_explanation)

# ========== Step 3: 应用推理规则，将每个 business_ref 映射为 business_id ==========
def resolve_ref_to_id(business_ref):
    prompt = (
        f"The previously inferred mapping rule is:\n\n{mapping_rule_explanation}\n\n"
        f"Now, based on this rule, please determine the corresponding `business_id` for the following `business_ref`:\n"
        f"- business_ref: {business_ref}\n\n"
        "Respond only with the `business_id`, e.g., 'biz_123'."
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a data assistant that maps review business_refs to true business_ids using the inferred rule."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on {business_ref}: {e}")
        return None

# ========== Step 4: 遍历所有 review 记录，逐条转换 ==========
df_review = con_duck.execute("SELECT * FROM review").fetchdf()

predicted_business_ids = []
for i, row in df_review.iterrows():
    business_ref = row["business_ref"]
    business_id = resolve_ref_to_id(business_ref)
    predicted_business_ids.append(business_id)
    print(f"[{i}] business_ref = {business_ref} → business_id = {business_id}")

df_review["business_id"] = predicted_business_ids



🧠 GPT-inferred mapping rule:

1. **Mapping Rule**: The mapping between `business_ref` and `business_id` is determined by the numeric portion of each ID. Specifically, the numeric part of `business_ref` (after the prefix "businessref_") corresponds directly to the numeric part of `business_id` (after the prefix "businessid_"). For example, `businessref_66` maps to `businessid_66`, `businessref_9` maps to `businessid_9`, and so on.

2. **Explanation**: This rule holds because both ID columns follow a consistent pattern where the prefix ("businessref_" or "businessid_") is followed by a numeric identifier. By extracting the numeric portion from each ID, we can see that they are identical in value. For instance:
   - `businessref_66` has a numeric part of 66, which matches the numeric part of `businessid_66`.
   - This pattern continues for all entries in both datasets, indicating a one-to-one correspondence based solely on the numeric identifiers. Thus, the mapping can be established by 

In [16]:
from pymongo import MongoClient

client_mongo = MongoClient("mongodb://localhost:27017/")
biz_collection = client_mongo["yelp_business"]["business"]

# 通过 address 或 description 字段模糊匹配 "Indianapolis"
indianapolis_businesses = biz_collection.find({
    "$or": [
        {"address": {"$regex": "indianapolis", "$options": "i"}},
        {"description": {"$regex": "indianapolis", "$options": "i"}}
    ]
})

# 提取符合条件的 business_id
ind_biz_ids = [biz["business_id"] for biz in indianapolis_businesses if "business_id" in biz]
# df_review 是你之前匹配过 business_id 的完整 review 表
df_ind_reviews = df_review[df_review["business_id"].isin(ind_biz_ids)].copy()

# 检查是否有 rating 字段，计算平均值
if "rating" in df_ind_reviews.columns:
    avg_rating = df_ind_reviews["rating"].mean()
    print(f"📊 Indianapolis 地区商家的平均 rating 为：{avg_rating}")
else:
    print("⚠️ rating 字段不存在，请确认字段名称是否正确")



📊 Indianapolis 地区商家的平均 rating 为：3.547008547008547


In [2]:
from pymongo import MongoClient
from openai import AzureOpenAI
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["yelp_business"]
collection = db["business"]

business_docs = list(collection.find())  # 你也可以用分页处理
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o-mini"

def extract_us_state(description):
    prompt = (
        "Based on the following business description, identify the U.S. state where this business is most likely located.\n"
        "Only respond with the full state name (e.g., 'California', 'Texas', 'New York'). If uncertain, respond with 'Unknown'.\n\n"
        f"Description: {description}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts U.S. state names from business descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=10
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error: {e}")
        return "Unknown"
# business_docs 已包含 business_id 和 description
state_records = []

for i, biz in enumerate(business_docs):
    desc = biz.get("description", "")
    if not desc:
        continue

    state = extract_us_state(desc)
    biz_id = biz.get("business_id")

    state_records.append({"business_id": biz_id, "state": state})
    print(f"[{i}] {desc} → {state}")
# 构建 state → business_id 的映射 DataFrame
df_state_map = pd.DataFrame(state_records)


[0] Located at 6901 Phelps Rd in Goleta, CA, this facility offers a nurturing environment for young learners, providing a range of services in Education, Elementary Schools, Child Care & Day Care, Local Services, Preschools, and Montessori Schools. → California
[1] Located at 9916 Clayton Rd in St. Louis, MO, this establishment offers a wide range of services, including Hair Salons, Beauty & Spas, Hair Stylists, Skin Care, Blow Dry/Out Services, and Makeup Artists. → Missouri
[2] Located at 11655 W Executive Dr in Boise, ID, this facility offers enthusiasts a premier destination for Gun/Rifle Ranges, Active Life. → Idaho
[3] Located at 1615 Pasadena Ave S, Ste 430 in Saint Petersburg, FL, this facility offers a range of services in Internal Medicine, Doctors, Health & Medical. → Florida
[4] Located at 9655 E US Hwy 36, Unit H in Avon, IN, this establishment offers a range of services including Nail Salons, Hair Removal, Beauty & Spas, and Waxing. → Indiana
[5] Located at 735 Dodecanese

In [4]:
import json
import duckdb
from pymongo import MongoClient
from openai import AzureOpenAI

deployment_name = "gpt-4o-mini"
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com"
)

# ========== Step 1: 加载两个 DuckDB 表中的 ID ==========
# business_ref from yelp_user.db
con_duck = duckdb.connect("../query_dataset/yelp_user.db")
business_refs = con_duck.execute("SELECT DISTINCT business_ref FROM review").fetchdf()["business_ref"].dropna().tolist()

# ========== Step 2: 从 MongoDB 读取 business_id ==========
client_mongo = MongoClient("mongodb://localhost:27017/")
business_ids = client_mongo["yelp_business"]["business"].distinct("business_id")
# === Step 2: Load business_ref (DuckDB) and business_id (MongoDB) ===
unique_business_refs = sorted(
    con_duck.execute("SELECT DISTINCT business_ref FROM review").fetchdf()["business_ref"].dropna().tolist()
)

client_mongo = MongoClient("mongodb://localhost:27017/")
all_business_ids = sorted(
    client_mongo["yelp_business"]["business"].distinct("business_id")
)

# === Step 3: Infer mapping rule using GPT ===
def get_mapping_rule(business_ids, business_refs):
    prompt = (
        "You are given two complete ID columns from two different datasets:\n"
        f"- The first column is `business_ref` from a review dataset: {json.dumps(business_refs, indent=2)}\n"
        f"- The second column is `business_id` from a business metadata dataset: {json.dumps(business_ids, indent=2)}\n\n"
        "Each business_ref corresponds to a business_id, but the mapping rule is not provided.\n"
        "Determine the mapping relationship between the two sets of IDs.\n"
        "Please respond with:\n"
        "1. The exact mapping rule in plain English\n"
        "2. An explanation of why you think this rule holds."
    )
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a data engineer analyzing two ID columns from different datasets to infer a mapping rule."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

mapping_rule_explanation = get_mapping_rule(all_business_ids, unique_business_refs)
print("\n🧠 GPT-inferred mapping rule:\n")
print(mapping_rule_explanation)

# === Step 4: Map business_ref → business_id with caching ===
def resolve_ref_to_id(business_ref):
    prompt = (
        f"The previously inferred mapping rule is:\n\n{mapping_rule_explanation}\n\n"
        f"Now, based on this rule, please determine the corresponding `business_id` for the following `business_ref`:\n"
        f"- business_ref: {business_ref}\n\n"
        "Respond only with the `business_id`, e.g., 'biz_123'."
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a data assistant that maps review business_refs to true business_ids using the inferred rule."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on {business_ref}: {e}")
        return None

# Read full review table
df_review = con_duck.execute("SELECT * FROM review").fetchdf()

# Create cache to avoid duplicate GPT calls
ref_to_id_cache = {}
predicted_business_ids = []

for i, row in df_review.iterrows():
    business_ref = row["business_ref"]
    if pd.isna(business_ref):
        predicted_business_ids.append(None)
        continue

    if business_ref not in ref_to_id_cache:
        business_id = resolve_ref_to_id(business_ref)
        ref_to_id_cache[business_ref] = business_id
    else:
        business_id = ref_to_id_cache[business_ref]

    predicted_business_ids.append(business_id)
    print(f"[{i}] business_ref = {business_ref} → business_id = {business_id}")

df_review["business_id"] = predicted_business_ids



🧠 GPT-inferred mapping rule:

1. **Mapping Rule**: Each `business_ref` ID corresponds to a `business_id` ID by replacing the prefix "businessref_" with "businessid_" while keeping the numeric part the same. For example, "businessref_1" maps to "businessid_1", "businessref_10" maps to "businessid_10", and so on.

2. **Explanation**: This rule holds because both ID columns follow a consistent pattern where the numeric portion of the ID is identical, and the only difference is the prefix. The prefixes "businessref_" and "businessid_" are systematically different but do not affect the numeric part, which is the key identifier. By analyzing the structure of both ID columns, it is clear that the numeric identifiers (1, 2, 3, ..., 100) are the same across both datasets, confirming that the mapping is based solely on the prefix change. This pattern is consistent across all entries in both lists, indicating a direct one-to-one correspondence between the two sets of IDs.
[0] business_ref = busi

In [5]:
import pandas as pd

# Merge state info into review data
df_review_with_state = pd.merge(df_review, df_state_map, on="business_id", how="left")

# Drop reviews with unknown or missing state
df_review_with_state = df_review_with_state.dropna(subset=["state"])

# Group by state to count reviews and compute average rating
df_state_stats = df_review_with_state.groupby("state").agg(
    num_reviews=("rating", "count"),
    avg_rating=("rating", "mean")
).reset_index()

# Sort to find the state with the most reviews
df_state_stats = df_state_stats.sort_values(by="num_reviews", ascending=False)
top_state = df_state_stats.iloc[0]

# Print result
print(f"🏆 State with most reviews: {top_state['state']}")
print(f"📝 Number of reviews: {top_state['num_reviews']}")
print(f"⭐ Average rating from that state: {top_state['avg_rating']:.2f}")


🏆 State with most reviews: Pennsylvania
📝 Number of reviews: 662
⭐ Average rating from that state: 3.70


In [10]:
import json
import duckdb
import pandas as pd
from pymongo import MongoClient
from openai import AzureOpenAI

# ========== Step 1: Setup MongoDB and DuckDB ==========
client_mongo = MongoClient("mongodb://localhost:27017/")
biz_collection = client_mongo["yelp_business"]["business"]

con_duck = duckdb.connect("../query_dataset/yelp_user.db")

# ========== Step 2: Setup Azure OpenAI ==========
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o-mini"

# ========== Step 3: Load business_ref and review table ==========
df_review = con_duck.execute("SELECT * FROM review").fetchdf()
unique_business_refs = df_review["business_ref"].dropna().unique().tolist()

# ========== Step 4: Load business_id and attributes from MongoDB ==========
biz_cursor = biz_collection.find({}, {"business_id": 1, "attributes": 1})
business_docs = list(biz_cursor)
all_business_ids = [doc["business_id"] for doc in business_docs if "business_id" in doc]

# ========== Step 5: GPT - Infer business_ref → business_id mapping ==========
def get_mapping_rule(business_ids, business_refs):
    prompt = (
        "You are given two complete ID columns from two different datasets:\n"
        f"- The first column is `business_ref` from a review dataset: {json.dumps(business_refs, indent=2)}\n"
        f"- The second column is `business_id` from a business metadata dataset: {json.dumps(business_ids, indent=2)}\n\n"
        "Each business_ref corresponds to a business_id, but the mapping rule is not provided.\n"
        "Determine the mapping relationship between the two sets of IDs.\n"
        "Please respond with:\n"
        "1. The exact mapping rule in plain English\n"
        "2. An explanation of why you think this rule holds."
    )
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a data engineer analyzing ID columns from different datasets to infer a mapping rule."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

mapping_rule_explanation = get_mapping_rule(all_business_ids, unique_business_refs)
print("\n🧠 Inferred Mapping Rule:\n", mapping_rule_explanation)

def resolve_ref_to_id(business_ref):
    prompt = (
        f"The inferred mapping rule is:\n\n{mapping_rule_explanation}\n\n"
        f"Now determine the business_id for:\n"
        f"- business_ref: {business_ref}\n\n"
        "Only respond with the business_id (e.g., 'biz_001')."
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a data assistant mapping business_ref to business_id."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on {business_ref}: {e}")
        return None

# Map all business_refs in review table
ref_to_id_map = {}
predicted_business_ids = []

for i, row in df_review.iterrows():
    business_ref = row["business_ref"]
    if pd.isna(business_ref):
        predicted_business_ids.append(None)
        continue
    if business_ref not in ref_to_id_map:
        ref_to_id_map[business_ref] = resolve_ref_to_id(business_ref)
    predicted_business_ids.append(ref_to_id_map[business_ref])
    print(f"[{i}] {business_ref} → {ref_to_id_map[business_ref]}")

df_review["business_id"] = predicted_business_ids

# ========== Step 6: GPT - Determine which businesses offer parking ==========
def offers_any_parking(attributes):
    prompt = (
    "Given the following business attributes, does the business offer either Business Parking or Bike Parking?\n"
    "Business Parking is considered available if **any** of the following are True: garage, street, validated, lot, valet.\n"
    "Respond only with 'yes' or 'no'.\n\n"
    "Note: BusinessParking may be a dictionary encoded as a string. Parse it before making a decision.\n\n"
    f"Attributes: {json.dumps(attributes, indent=2)}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You check whether businesses offer parking from attributes."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower() == "yes"
    except Exception as e:
        print(f"❌ GPT error on attributes: {e}")
        return False

parking_business_ids = []
for i, biz in enumerate(business_docs):
    biz_id = biz.get("business_id")
    attrs = biz.get("attributes", {})
    if not isinstance(attrs, dict):
        continue
    if offers_any_parking(attrs):
        parking_business_ids.append(biz_id)
        print(f"✅ [{i}] {attrs} → offers Parking")
    else:
        print(f"❌ [{i}] {attrs} → no Parking")

# ========== Step 7: Filter reviews from 2018 ==========
df_review["date"] = pd.to_datetime(df_review["date"], unit="ms")
df_2018 = df_review[df_review["date"].dt.year == 2018].copy()


# ========== Step 8: Answer query ==========
df_2018_with_parking = df_2018[df_2018["business_id"].isin(parking_business_ids)]
num_businesses = df_2018_with_parking["business_id"].nunique()

print(f"\n📊 In 2018, number of businesses reviewed that offered Business or Bike Parking: {num_businesses}")




🧠 Inferred Mapping Rule:
 1. **Mapping Rule**: The mapping between `business_ref` and `business_id` is determined by the numeric portion of each ID. Specifically, the numeric part of `business_ref` (after "businessref_") corresponds to the numeric part of `business_id` (after "businessid_") in a one-to-one relationship. For example, `businessref_34` maps to `businessid_34`, `businessref_1` maps to `businessid_1`, and so on.

2. **Explanation**: This rule holds because both ID columns contain a consistent prefix followed by a numeric identifier. By examining the numeric portions of the IDs, we can see that they are identical for corresponding entries. For instance, if we take `business_ref` with the numeric part `34`, it directly corresponds to `business_id` with the same numeric part `34`. This pattern continues throughout the entire dataset, indicating a direct mapping based solely on the numeric identifiers. The structure of the IDs suggests that they are designed to be related in t

In [7]:
import duckdb
import pandas as pd

# Step 1: 连接并读取 review 表中的 date 字段
con = duckdb.connect("../query_dataset/yelp_user.db")
df = con.execute("SELECT date FROM review LIMIT 5").fetchdf()

# Step 2: 显示数据和类型
print("🔎 前5条日期值：")
print(df["date"])

print("\n📦 数据类型：")
print(df["date"].dtype)

# Step 3: 尝试使用 .str.startswith("2018")
print("\n✅ 尝试 str.startswith('2018'):")
try:
    print(df["date"].str.startswith("2018"))
except Exception as e:
    print(f"❌ 失败：{e}")

# Step 4: 尝试转为 datetime 再取年份
print("\n✅ 尝试 pd.to_datetime → year == 2018:")
try:
    df["date_parsed"] = pd.to_datetime(df["date"])
    print(df["date_parsed"].dt.year == 2018)
except Exception as e:
    print(f"❌ 失败：{e}")


🔎 前5条日期值：
0    1470023040000
1    1623670740000
2    1369868460000
3    1463856532000
4    1635786719000
Name: date, dtype: int64

📦 数据类型：
int64

✅ 尝试 str.startswith('2018'):
❌ 失败：Can only use .str accessor with string values!

✅ 尝试 pd.to_datetime → year == 2018:
0    False
1    False
2    False
3    False
4    False
Name: date_parsed, dtype: bool


In [1]:
from pymongo import MongoClient
from openai import AzureOpenAI
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["yelp_business"]
collection = db["business"]

business_docs = list(collection.find())  # 你也可以用分页处理
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o-mini"

def extract_categories(description):
    prompt = (
        "Based on the following business description, identify one or more likely business categories.\n"
        "Respond with a comma-separated list of categories (e.g., 'Nail Salon, Spa', 'Bar, Restaurant'). "
        "Avoid vague responses like 'business' or 'store'.\n\n"
        f"Description: {description}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts specific business categories from descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=30
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on category extraction: {e}")
        return "Unknown"
category_records = []

for i, biz in enumerate(business_docs):
    desc = biz.get("description", "")
    if not desc:
        continue

    biz_id = biz.get("business_id")
    category_str = extract_categories(desc)

    if category_str.lower() == "unknown":
        continue

    categories = [cat.strip() for cat in category_str.split(",") if cat.strip()]
    for cat in categories:
        category_records.append({"business_id": biz_id, "category": cat})

    print(f"[{i}] {desc} → {category_str}")
df_category_map = pd.DataFrame(category_records)


[0] Located at 6901 Phelps Rd in Goleta, CA, this facility offers a nurturing environment for young learners, providing a range of services in Education, Elementary Schools, Child Care & Day Care, Local Services, Preschools, and Montessori Schools. → Education, Child Care & Day Care, Preschools, Montessori Schools
[1] Located at 9916 Clayton Rd in St. Louis, MO, this establishment offers a wide range of services, including Hair Salons, Beauty & Spas, Hair Stylists, Skin Care, Blow Dry/Out Services, and Makeup Artists. → Hair Salon, Beauty Spa, Skin Care, Makeup Artist
[2] Located at 11655 W Executive Dr in Boise, ID, this facility offers enthusiasts a premier destination for Gun/Rifle Ranges, Active Life. → Gun Range, Outdoor Recreation
[3] Located at 1615 Pasadena Ave S, Ste 430 in Saint Petersburg, FL, this facility offers a range of services in Internal Medicine, Doctors, Health & Medical. → Internal Medicine, Healthcare, Medical Clinic
[4] Located at 9655 E US Hwy 36, Unit H in Avo